# Spotify Listening Analysis
**Question:** How do I listen to music, what makes me skip, and can I predict a skip before it happens?

Run each cell from top to bottom (Shift+Enter). Cells marked **📝 YOUR NOTES** are for you to write in.
Those notes become your README and your interview answers.

## 0. Setup
If an import fails, run this in the VS Code terminal (with your `.venv` activated):
`pip install pandas matplotlib seaborn scikit-learn`

In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

# ---- SETTINGS: the only things you should need to change ----
DATA_PATH = "data/spotify_history.csv"   # Maven CSV. For YOUR data later: "data/Streaming_History_Audio_*.json"
TIMEZONE = None            # Set to "Europe/London" for your own data (Spotify timestamps are UTC). Leave None for Maven.
MIN_STREAM_MS = 30_000     # Spotify counts a play as a "stream" at 30 seconds
MIN_PLAYS_PER_ARTIST = 20  # ignore artists with too few plays when ranking skip rates
SESSION_GAP_MIN = 30       # a gap longer than this starts a new listening session

## 1. Load the data
This cell works with **both** the Maven CSV and Spotify's own JSON export, so it'll still work when your own data arrives.

In [ ]:
files = sorted(glob.glob(DATA_PATH))
if not files:
    raise FileNotFoundError(f"No files match {DATA_PATH}. Check the file name and that it's in the data folder.")

if files[0].endswith(".json"):
    raw = pd.concat([pd.read_json(f) for f in files], ignore_index=True)
else:
    raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print(f"{len(files)} file(s), {raw.shape[0]:,} rows, {raw.shape[1]} columns")
raw.head()

In [ ]:
raw.info()

### Standardise column names
Spotify's export and the Maven CSV name some columns differently. This maps both to one set of names.

In [ ]:
rename_map = {
    # Spotify extended-history JSON names -> our names
    "master_metadata_track_name": "track_name",
    "master_metadata_album_artist_name": "artist_name",
    "master_metadata_album_album_name": "album_name",
    # Spotify basic "Account data" names -> our names
    "endTime": "ts", "artistName": "artist_name", "trackName": "track_name", "msPlayed": "ms_played",
}
df = raw.rename(columns=rename_map).copy()

# Personal/identifying fields we never want in the analysis or on GitHub
PRIVATE_COLS = ["ip_addr", "ip_addr_decrypted", "user_agent_decrypted", "username", "conn_country"]
df = df.drop(columns=[c for c in PRIVATE_COLS if c in df.columns])

print("Columns now:", list(df.columns))

## 2. Explore before cleaning
Look at what's messy *before* you change anything.

In [ ]:
print("Missing values per column:")
print(df.isna().sum().sort_values(ascending=False))

In [ ]:
for col in ["platform", "reason_start", "reason_end", "shuffle", "skipped"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts(dropna=False).head(10))

In [ ]:
df["ms_played"].describe()

### 📝 YOUR NOTES: what looks messy?
- (e.g. how many rows have no track name? any weird platform values? are `skipped` values blank for some years?)

## 3. Clean

In [ ]:
n0 = len(df)
clean_log = []   # every cleaning decision is recorded here and printed at the end

# 3a. Remove rows with no track or artist (podcasts, or Spotify's known missing-metadata rows)
before = len(df)
df = df.dropna(subset=["track_name", "artist_name"])
clean_log.append(f"Dropped {before - len(df):,} rows with no track/artist name")

# 3b. Timestamps -> datetime (and convert to local time if set)
df["ts"] = pd.to_datetime(df["ts"], utc=True)
if TIMEZONE:
    df["ts"] = df["ts"].dt.tz_convert(TIMEZONE)
    clean_log.append(f"Converted timestamps from UTC to {TIMEZONE}")
df["ts"] = df["ts"].dt.tz_localize(None)

# 3c. Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates()
clean_log.append(f"Dropped {before - len(df):,} exact duplicate rows")

# 3d. Make shuffle / skipped proper True/False (blank skipped -> False, but we note how many)
def to_bool(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "1.0", "yes"])
if "skipped" in df.columns:
    clean_log.append(f"{df['skipped'].isna().sum():,} rows had a blank 'skipped' value (treated as not skipped)")
    df["skipped"] = to_bool(df["skipped"])
if "shuffle" in df.columns:
    df["shuffle"] = to_bool(df["shuffle"])

# 3e. Sort by time
df = df.sort_values("ts").reset_index(drop=True)

print(f"Rows: {n0:,} -> {len(df):,}")
for line in clean_log:
    print(" -", line)

## 4. Define metrics and add features
These definitions are the kind of thing interviewers ask about, so make sure you can explain each one.

In [ ]:
# A "stream" = played for at least 30 seconds (Spotify's own rule)
df["is_stream"] = df["ms_played"] >= MIN_STREAM_MS

# A "skip" = Spotify's skipped flag OR the track ended because the forward button was pressed
skip = df["skipped"] if "skipped" in df.columns else pd.Series(False, index=df.index)
if "reason_end" in df.columns:
    skip = skip | (df["reason_end"] == "fwdbtn")
df["is_skip"] = skip

# Time features
df["date"] = df["ts"].dt.date
df["hour"] = df["ts"].dt.hour
df["weekday"] = df["ts"].dt.day_name()
df["month"] = df["ts"].dt.to_period("M").astype(str)
df["minutes_played"] = df["ms_played"] / 60_000

# Familiarity features: only use what was known BEFORE each play (no peeking into the future)
df["track_prior_plays"] = df.groupby("track_name").cumcount()
df["artist_prior_plays"] = df.groupby("artist_name").cumcount()
df["artist_is_new"] = df["artist_prior_plays"] == 0

# Sessions: a new session starts after a gap of more than SESSION_GAP_MIN minutes
gap = df["ts"].diff().dt.total_seconds().div(60)
df["session_id"] = (gap.isna() | (gap > SESSION_GAP_MIN)).cumsum()
df["position_in_session"] = df.groupby("session_id").cumcount()

df[["ts", "artist_name", "track_name", "minutes_played", "is_stream", "is_skip",
    "artist_is_new", "session_id", "position_in_session"]].head(10)

In [ ]:
# Save the cleaned data, which is what Tableau will use
import os
os.makedirs("output", exist_ok=True)
df.to_csv("output/cleaned.csv", index=False)
print("Saved output/cleaned.csv")

## 5. Analyse

### 5a. Headline numbers (your dashboard KPIs)

In [ ]:
kpis = {
    "Date range": f"{df['ts'].min():%d %b %Y} to {df['ts'].max():%d %b %Y}",
    "Total plays": f"{len(df):,}",
    "Total hours listened": f"{df['minutes_played'].sum() / 60:,.0f}",
    "Unique artists": f"{df['artist_name'].nunique():,}",
    "Unique tracks": f"{df['track_name'].nunique():,}",
    "Skip rate": f"{df['is_skip'].mean():.1%}",
    "Plays under 30s": f"{(~df['is_stream']).mean():.1%}",
    "Listening sessions": f"{df['session_id'].nunique():,}",
}
pd.Series(kpis, name="value").to_frame()

### 5b. When do I listen? (heatmap)

In [ ]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = (df.pivot_table(index="weekday", columns="hour", values="minutes_played", aggfunc="sum")
          .reindex(days).fillna(0) / 60)

plt.figure(figsize=(14, 4))
sns.heatmap(heat, cmap="Greens", cbar_kws={"label": "hours listened"})
plt.title("Listening hours by weekday and hour")
plt.xlabel("Hour of day"); plt.ylabel("")
plt.tight_layout(); plt.show()

### 5c. When do I skip most?

In [ ]:
skip_by_hour = df.groupby("hour")["is_skip"].mean()
ax = skip_by_hour.plot(kind="bar", figsize=(12, 4), color="#4C72B0")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
plt.title("Skip rate by hour of day"); plt.xlabel("Hour"); plt.ylabel("Skip rate")
plt.tight_layout(); plt.show()

if "shuffle" in df.columns:
    print("Skip rate with shuffle ON vs OFF:")
    print(df.groupby("shuffle")["is_skip"].mean().map("{:.1%}".format))
print("\nSkip rate for a NEW artist vs a known artist:")
print(df.groupby("artist_is_new")["is_skip"].mean().map("{:.1%}".format))

### 5d. Which artists do I finish vs abandon?

In [ ]:
artist_stats = (df.groupby("artist_name")
                  .agg(plays=("is_skip", "size"), skip_rate=("is_skip", "mean"),
                       hours=("minutes_played", lambda m: m.sum() / 60))
                  .query("plays >= @MIN_PLAYS_PER_ARTIST"))

print(f"{len(artist_stats)} artists have at least {MIN_PLAYS_PER_ARTIST} plays")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
artist_stats.nsmallest(10, "skip_rate")["skip_rate"].sort_values().plot(kind="barh", ax=axes[0], color="#55A868")
axes[0].set_title("Least skipped artists")
artist_stats.nlargest(10, "skip_rate")["skip_rate"].sort_values().plot(kind="barh", ax=axes[1], color="#C44E52")
axes[1].set_title("Most skipped artists")
for ax in axes:
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1)); ax.set_ylabel("")
plt.tight_layout(); plt.show()

### 5e. Discovery and retention: do new artists stick?

In [ ]:
first_play = df.groupby("artist_name")["ts"].min().rename("first_play")
last_play = df.groupby("artist_name")["ts"].max().rename("last_play")
artists = pd.concat([first_play, last_play], axis=1)

# New artists discovered per month
new_per_month = artists["first_play"].dt.to_period("M").value_counts().sort_index()
new_per_month.index = new_per_month.index.astype(str)
new_per_month.plot(figsize=(12, 4), marker="o")
plt.title("New artists discovered per month"); plt.ylabel("artists"); plt.xlabel("")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# Retention: of artists discovered at least 90 days before the data ends,
# what share were still being played 90+ days after first listen?
cutoff = df["ts"].max() - pd.Timedelta(days=90)
eligible = artists[artists["first_play"] <= cutoff]
retained = (eligible["last_play"] - eligible["first_play"]) >= pd.Timedelta(days=90)
if len(eligible):
    print(f"90-day artist retention: {retained.mean():.1%} of {len(eligible):,} discovered artists")
else:
    print("Not enough history for 90-day retention (need data spanning more than 90 days).")

### 📝 YOUR NOTES: findings (pick the best 3 to 5)
Write each as **insight + so what**. Example:
- *Insight:* skip rate is highest between 7 and 9am (X%). *So what:* morning recommendations should favour familiar tracks.
1.
2.
3.

## 6. Model: can I predict a skip? (the AI/ML part)

**Avoid data leakage:** the model may only use information known *when the track starts*.
So `ms_played` and `reason_end` are **not** allowed as inputs, because they tell you the answer.
Being able to explain this is one of the strongest things you can say in an interview.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score

numeric = ["hour", "track_prior_plays", "artist_prior_plays", "position_in_session"]
flags = [c for c in ["shuffle", "artist_is_new"] if c in df.columns]
categorical = [c for c in ["weekday", "platform", "reason_start"] if c in df.columns]

X = pd.get_dummies(df[numeric + flags + categorical], columns=categorical, drop_first=True).astype(float)
y = df["is_skip"].astype(int)

print(f"Features: {X.shape[1]}  |  Rows: {len(X):,}")
print(f"Share of plays that are skips: {y.mean():.1%}")
baseline = max(y.mean(), 1 - y.mean())
print(f"A model that always guesses the most common outcome would already be {baseline:.1%} accurate, so accuracy alone is misleading.")

In [ ]:
# Train on the earlier 80% of plays, test on the latest 20%: predicting the future from the past.
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

models = {
    "Logistic regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random forest": RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced",
                                            random_state=42, n_jobs=-1),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    print(f"\n===== {name} =====  ROC AUC: {roc_auc_score(y_test, prob):.3f}")
    print(classification_report(y_test, pred, target_names=["not skipped", "skipped"], digits=3))

In [ ]:
rf = models["Random forest"]
ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, display_labels=["not skipped", "skipped"], cmap="Blues")
plt.title("Random forest: confusion matrix (test set)"); plt.show()

importance = pd.Series(rf.feature_importances_, index=X.columns).nlargest(12).sort_values()
importance.plot(kind="barh", figsize=(9, 5), color="#8172B2")
plt.title("What predicts a skip? (random forest feature importance)")
plt.tight_layout(); plt.show()

### 📝 YOUR NOTES: model
- Which model did better, and on which metric? (focus on **recall** and **precision** for "skipped", plus ROC AUC: 0.5 is random guessing, 1.0 is perfect)
- What were the top 3 drivers of skipping?
- What would you add with more time? (e.g. genre, track length, time since last play of that artist)

## 7. Cleaning log (copy into your README)

In [ ]:
print(f"Started with {n0:,} rows, finished with {len(df):,}.")
for line in clean_log:
    print(" -", line)
print(f" - Defined a stream as >= {MIN_STREAM_MS // 1000}s played")
print(" - Defined a skip as skipped == True OR reason_end == 'fwdbtn'")
print(f" - Sessions split on gaps longer than {SESSION_GAP_MIN} minutes")
print(" - Removed personal fields (IP address, username, etc.)")